<img src="https://img.shields.io/badge/cloudwithshad-Week%203-00B4D8?style=for-the-badge" />

# How Chatbot Memory Really Works
### The illusion of memory, token budgets, and trimming history like a pro

**cloudwithshad** · *Build Your First AI App — Python from Zero* · **Deep Dive 1 of 2** · ⏱ ~50 min · ✅ No API key needed — runs 100% free

---
**How to use this notebook**
- 📓 Open it in **Google Colab** (easiest — nothing to install) or Jupyter/VS Code.
- ▶️ Run every cell yourself (Shift+Enter). Reading is not learning — running is.
- ✏️ Cells marked **🧪 TRY IT** are safe to change. Break things on purpose; that's how you learn.


## 1 · The secret: the AI remembers NOTHING 🤯

Here's what the lab hinted at but didn't fully say: **the model has zero memory between calls.**
Every single request starts from a blank slate. "Memory" is an illusion **you** create by re-sending
the whole conversation every time.

Let's simulate a conversation the *broken* way (no history) vs the *right* way — with no API,
so you can see the mechanics clearly:

In [ ]:
# The BROKEN way — each call sends only the newest message:
call_1 = [{"role": "user", "content": "My name is Kofi."}]
call_2 = [{"role": "user", "content": "What is my name?"}]   # ⚠️ the AI sees ONLY this!
print("Call 2 sends:", call_2)
print("➡️ The AI has no idea. It never saw call 1.")

In [ ]:
# The RIGHT way — memory is a growing list; every call sends ALL of it:
memory = [{"role": "system", "content": "You are helpful."}]

memory.append({"role": "user", "content": "My name is Kofi."})
memory.append({"role": "assistant", "content": "Nice to meet you, Kofi!"})
memory.append({"role": "user", "content": "What is my name?"})

print(f"This call sends {len(memory)} messages:")
for m in memory:
    print(f'  [{m["role"]}] {m["content"]}')
print("➡️ Now the model can see 'My name is Kofi' right there in the request.")

**That's the whole trick.** `st.session_state.messages` in your lab is exactly this `memory` list —
Streamlit's backpack just keeps it alive between re-runs.

## 2 · The full chat loop, in plain Python

Strip away Streamlit and a chatbot is ~10 lines. Read this until it feels obvious:

In [ ]:
def fake_ai(messages):
    """Pretend AI so we can study the LOOP without spending tokens."""
    last = messages[-1]["content"]
    names = [m["content"] for m in messages if "name is" in m["content"].lower()]
    if "my name" in last.lower() and names:
        return f"You told me earlier: {names[0]}"
    return f"(I received {len(messages)} messages of context)"

memory = [{"role": "system", "content": "You are helpful."}]

for user_text in ["My name is Kofi.", "I live in Tema.", "What is my name?"]:
    memory.append({"role": "user", "content": user_text})       # 1. remember input
    reply = fake_ai(memory)                                     # 2. send EVERYTHING
    memory.append({"role": "assistant", "content": reply})      # 3. remember reply
    print(f"👤 {user_text}\n🤖 {reply}\n")

The three-step rhythm — **append user ➜ send all ➜ append assistant** — is every chatbot ever,
including ChatGPT itself.

## 3 · The problem memory creates: cost & limits 💸

If you re-send everything every time, long chats get expensive — and models have a **context limit**
(a maximum number of tokens per request). Let's *see* the growth:

In [ ]:
def rough_tokens(messages):
    """Rough estimate: ~1 token per 4 characters of English."""
    return sum(len(m["content"]) for m in messages) // 4

memory = [{"role": "system", "content": "You are a helpful assistant for Ghanaian students."}]
for turn in range(1, 21):
    memory.append({"role": "user", "content": f"This is my question number {turn}, and it is reasonably long like real questions are."})
    memory.append({"role": "assistant", "content": "Here is a helpful, fairly detailed answer that takes a few lines to say properly." * 2})
    if turn % 5 == 0:
        print(f"After {turn} turns: {len(memory)} messages ≈ {rough_tokens(memory)} tokens PER CALL")

Every new turn re-pays for **all previous turns**. A 100-turn chat can cost more *per message*
than your first 20 chats combined.

## 4 · The fix: trim the history ✂️

Pros keep the **system message** (never lose the personality!) plus the **last N messages**:

In [ ]:
def trim_history(messages, keep_last=6):
    """Keep the system message + the last `keep_last` messages."""
    system = [m for m in messages if m["role"] == "system"]
    rest = [m for m in messages if m["role"] != "system"]
    return system + rest[-keep_last:]

trimmed = trim_history(memory, keep_last=6)
print(f"Before: {len(memory)} messages ≈ {rough_tokens(memory)} tokens")
print(f"After:  {len(trimmed)} messages ≈ {rough_tokens(trimmed)} tokens")
print("First kept message role:", trimmed[0]["role"], "← personality survives!")

**Trade-off:** the bot forgets old details (like a human!) but stays fast and cheap. For a
smarter upgrade, real products *summarize* old messages instead of dropping them — a great
capstone stretch goal.

## 5 · Drop-in upgrade for your lab chatbot

In your Week 3 `app.py`, change one line to make your bot production-ready:

```python
response = client.chat.completions.create(
    model=MODEL,
    messages=trim_history(st.session_state.messages, keep_last=12),  # ← was: st.session_state.messages
)
```

(And paste the `trim_history` function near the top.) Same bot, but it can now chat forever
without your bill growing forever.

## 🎓 Recap

| Truth | Consequence |
|---|---|
| The model remembers nothing | *You* re-send history every call |
| Memory = list of dicts | `.append()` after every turn (user AND assistant) |
| History re-sent each call | Cost grows with chat length |
| Trim: system + last N | Personality survives, bill doesn't explode |
